# Tool-Calling Agent in 2 Hours

Fine-tune Qwen 3.5 0.8B into a function-calling agent using StateSet Agents GSPO + `ToolAgent`. The third leg of the framework's three-pillar showcase, alongside `whitepaper_v1_gsm8k_benchmark.ipynb` (math reasoning) and `customer_support_4h.ipynb` (multi-turn dialogue).

**Estimated runtime:** ~2 hours on a Colab A100. **Cost:** ~$1.20.

## What this notebook does

1. Pins the framework to commit `14c0e65`.
2. Sets seed `42` across all RNGs.
3. Registers three sample tools: `get_weather`, `calculator`, `search`.
4. Loads 6 scenarios with `expected_tool` + `expected_params` + `expected_outcome` per row.
5. Evaluates the un-fine-tuned baseline against the composite tool-call reward.
6. Fine-tunes with **GSPO** using `train_with_gspo`.
7. Re-evaluates and saves a schema-compliant JSON result.

**Open in Colab:** [![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/stateset/stateset-agents/blob/master/notebooks/tool_calling_agent_demo.ipynb)

Set runtime to A100 for the published timings.

## 1. Pin + install

In [ ]:
import os
import subprocess

PINNED_COMMIT = '14c0e65'

if not os.path.exists('/content/stateset-agents'):
    subprocess.check_call([
        'git', 'clone', '--quiet',
        'https://github.com/stateset/stateset-agents',
        '/content/stateset-agents'
    ])
subprocess.check_call(['git', '-C', '/content/stateset-agents', 'checkout', '--quiet', PINNED_COMMIT])
%cd /content/stateset-agents
print('Pinned to', subprocess.check_output(['git', 'rev-parse', '--short', 'HEAD']).decode().strip())

In [ ]:
%pip install --quiet -e '.[training]'
%pip install --quiet accelerate bitsandbytes
print('Install complete')

## 2. Seeds + tool registry

In [ ]:
from stateset_agents.utils.reproducibility import set_all_seeds

SEED = 42
state = set_all_seeds(SEED)
print('Seeds applied:', state.to_dict())

SAMPLE_TOOLS = [
    {
        'name': 'get_weather',
        'description': 'Get the current weather for a city.',
        'parameters': {'type': 'object', 'properties': {'city': {'type': 'string'}}, 'required': ['city']},
    },
    {
        'name': 'calculator',
        'description': 'Evaluate a math expression.',
        'parameters': {'type': 'object', 'properties': {'expression': {'type': 'string'}}, 'required': ['expression']},
    },
    {
        'name': 'search',
        'description': 'Search a knowledge base.',
        'parameters': {'type': 'object', 'properties': {'query': {'type': 'string'}}, 'required': ['query']},
    },
]

TRAIN_SCENARIOS = [
    {'user_query': "What's the weather in San Francisco?", 'expected_tool': 'get_weather', 'expected_params': {'city': 'San Francisco'}, 'expected_outcome': '63'},
    {'user_query': 'Calculate 17 * 24', 'expected_tool': 'calculator', 'expected_params': {'expression': '17 * 24'}, 'expected_outcome': '408'},
    {'user_query': 'Look up the population of Tokyo', 'expected_tool': 'search', 'expected_params': {'query': 'population of Tokyo'}, 'expected_outcome': '13.96'},
    {'user_query': 'Find recent papers on diffusion models', 'expected_tool': 'search', 'expected_params': {'query': 'diffusion models'}, 'expected_outcome': 'papers'},
]
EVAL_SCENARIOS = [
    {'user_query': 'Calculate the square root of 144', 'expected_tool': 'calculator', 'expected_params': {'expression': 'sqrt(144)'}, 'expected_outcome': '12'},
    {'user_query': "What's the weather forecast for Paris tomorrow?", 'expected_tool': 'get_weather', 'expected_params': {'city': 'Paris'}, 'expected_outcome': '58'},
]
print(f'{len(TRAIN_SCENARIOS)} train, {len(EVAL_SCENARIOS)} eval scenarios')

## 3. Tool-call reward (composite)

Three signals: tool selection, parameter correctness, outcome substring match. This is the same reward shipped in the `tool-calling-agent` starter template.

In [ ]:
import json
import re
from typing import Any

from stateset_agents.core.reward_base import RewardFunction, RewardResult, RewardType
from stateset_agents.core.trajectory import ConversationTurn

_TOOL_BLOCK_RE = re.compile(r'```json\s*(\{.*?\})\s*```', re.DOTALL)

def _extract_tool_call(response: str):
    m = _TOOL_BLOCK_RE.search(response or '')
    if not m:
        return None
    try:
        d = json.loads(m.group(1))
    except json.JSONDecodeError:
        return None
    return d if isinstance(d, dict) and 'tool' in d else None

class ToolCallReward(RewardFunction):
    name = 'tool_call_composite'
    def __init__(self):
        super().__init__(weight=1.0, reward_type=RewardType.SPARSE, name=self.name)
    async def compute_reward(self, turns, context=None):
        if not turns:
            return RewardResult(score=0.0)
        full = '\n'.join(t.content for t in turns if t.role == 'assistant' and t.content)
        ctx = context or {}
        call = _extract_tool_call(full)
        tool_score = 1.0 if call and call.get('tool', '') == ctx.get('expected_tool') else 0.0
        exp_params = ctx.get('expected_params', {})
        if call and exp_params:
            params = call.get('parameters', {}) or {}
            matched = sum(1 for k, v in exp_params.items() if str(params.get(k, '')).strip().lower() == str(v).strip().lower())
            param_score = matched / max(len(exp_params), 1)
        else:
            param_score = 1.0 if not exp_params else 0.0
        exp_out = str(ctx.get('expected_outcome', ''))
        outcome_score = 1.0 if (not exp_out or exp_out.lower() in full.lower()) else 0.0
        composite = 0.4 * tool_score + 0.3 * param_score + 0.3 * outcome_score
        return RewardResult(score=float(composite), breakdown={'tool_selection': tool_score, 'param_correctness': param_score, 'outcome': outcome_score})

## 4. Baseline evaluation

In [ ]:
import asyncio
from stateset_agents.core.tool_agent import ToolAgent
from stateset_agents.core.agent_config import AgentConfig

MODEL_NAME = 'Qwen/Qwen3.5-0.8B'

async def evaluate(agent, scenarios):
    reward = ToolCallReward()
    scores = []
    for s in scenarios:
        response = await agent.generate_response(
            f"User wants you to use a tool to help. Available tools have already been described.\nUser: {s['user_query']}\nRespond with a JSON tool call in a ```json ... ``` block.\nAgent:"
        )
        turns = [ConversationTurn(role='assistant', content=response)]
        result = await reward.compute_reward(turns, context=s)
        scores.append(result.score)
    return sum(scores) / max(len(scores), 1), scores

baseline_agent = ToolAgent(
    config=AgentConfig(model_name=MODEL_NAME, max_new_tokens=320, temperature=0.0, do_sample=False, torch_dtype='bfloat16'),
    tools=SAMPLE_TOOLS,
)
await baseline_agent.initialize()
baseline_score, baseline_per = await evaluate(baseline_agent, EVAL_SCENARIOS)
print(f'Baseline composite score: {baseline_score:.3f}')
for s, v in zip(EVAL_SCENARIOS, baseline_per):
    print(f'  {v:.2f}  expected={s["expected_tool"]}  query={s["user_query"][:50]}')

## 5. Fine-tune with GSPO

In [ ]:
from stateset_agents.training import GSPOConfig, train_with_gspo
from stateset_agents.core import ConversationEnvironment
import time

config = GSPOConfig(
    model_name=MODEL_NAME,
    output_dir='/content/gspo_tools',
    report_to='none',
    num_outer_iterations=4,
    num_iterations=1,
    num_generations=4,
    generations_per_iteration=len(TRAIN_SCENARIOS),
    clip_range_left=3e-4,
    clip_range_right=4e-4,
    learning_rate=5e-6,
    max_prompt_length=768,
    max_completion_length=320,
    use_lora=True,
    lora_r=16,
    lora_alpha=32,
    gradient_checkpointing=True,
    bf16=True,
    warmup_ratio=0.1,
)

# Do NOT call agent.initialize() — train_with_gspo loads the model itself.
trainer_agent = ToolAgent(
    config=AgentConfig(model_name=MODEL_NAME, max_new_tokens=320),
    tools=SAMPLE_TOOLS,
)

env = ConversationEnvironment(
    scenarios=TRAIN_SCENARIOS,
    reward_fn=ToolCallReward(),
    max_turns=1,
)

t0 = time.time()
trained_agent = await train_with_gspo(
    config=config,
    agent=trainer_agent,
    environment=env,
    reward_model=ToolCallReward(),
)
train_wall_clock = time.time() - t0
print(f'\nTraining wall-clock: {train_wall_clock:.0f}s ({train_wall_clock/60:.1f}m)')

## 6. Post-training evaluation + save result

In [ ]:
import torch
from datetime import datetime, timezone
from pathlib import Path

final_score, final_per = await evaluate(trained_agent, EVAL_SCENARIOS)
print(f'Final: {final_score:.3f}  (Δ {final_score - baseline_score:+.3f})')

result = {
    'trainer': 'gspo',
    'task': 'tool_calling',
    'model': MODEL_NAME,
    'seed': SEED,
    'commit': PINNED_COMMIT,
    'timestamp': datetime.now(timezone.utc).isoformat(),
    'config': {'num_generations': config.num_generations, 'learning_rate': config.learning_rate, 'lora_r': config.lora_r},
    'metrics': {
        'eval_pass_at_1': final_score,
        'eval_pass_at_1_baseline': baseline_score,
        'improvement': final_score - baseline_score,
        'wall_clock_seconds': train_wall_clock,
        'train_examples': len(TRAIN_SCENARIOS),
        'eval_examples': len(EVAL_SCENARIOS),
        'peak_vram_mb': torch.cuda.max_memory_allocated() // (1024**2) if torch.cuda.is_available() else 0,
    },
    'hardware': {'gpu': torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'cpu'},
}
out = Path(f'/content/gspo_seed{SEED}_tool_calling_qwen3_5_0_8b.json')
out.write_text(json.dumps(result, indent=2))
print(json.dumps(result, indent=2))

## 7. Next steps

- **Scale tools.** Replace the three stub tools with real API clients — Stripe, Slack, your CRM. The reward function works against any tool registry.
- **Scale scenarios.** 6 scenarios is a smoke test; 200+ for production. Same JSONL schema.
- **Run 3 seeds (42, 1337, 2026)** and aggregate via `make benchmark-aggregate`.
- **Scaffold a full project** with `stateset-agents starter tool-calling-agent ./my-project`.